# Robust costs – Adaptive IRLS

Using M-estimators with **adaptive scale estimation** (`irls_huber_adaptive`, `irls_cauchy_adaptive`, `irls_tukey_adaptive`) to handle outliers robustly in jaxls.

The fixed-parameter estimators (`irls_huber`, `irls_cauchy`, …) require you to choose a scale threshold in the same units as your residuals.  The adaptive variants remove this burden: at every solver iteration they estimate the noise scale **σ** from the current residuals using the Median Absolute Deviation (MAD),

$$\hat{\sigma} = \frac{\operatorname{median}(|r|)}{0.6745}$$

and then compute weights from the scale-normalised residuals $u_i = r_i / \hat{\sigma}$.  Because the scale is re-estimated each iteration, the estimator adapts to whatever noise level is present in the data—no manual tuning required.

Features used:
- {class}`~jaxls.Var` subclass for circle parameters
- {func}`@jaxls.Cost.factory <jaxls.Cost.factory>` with `irls_weight_fn` for robust residuals
- Adaptive M-estimator factories `jaxls.utils.irls_huber_adaptive`, `irls_cauchy_adaptive`, `irls_tukey_adaptive`

In [ ]:
import sys
from loguru import logger

logger.remove()
logger.add(sys.stdout, format="<level>{level: <8}</level> | {message}");

In [ ]:
import jax
import jax.numpy as jnp
import jaxls
import numpy as np

## The outlier problem

Consider fitting a circle to 2D points. With clean data, least squares works well.
But real-world data often contains outliers -- points that don't follow the expected
model due to sensor errors, misassociations, or other anomalies.

Standard least squares minimizes the sum of squared residuals:

$$\min_\theta \sum_i r_i(\theta)^2$$

The squaring amplifies large residuals, giving outliers disproportionate influence.

In [ ]:
# Generate synthetic circle data with outliers.
np.random.seed(42)

# Ground truth circle.
true_cx, true_cy, true_r = 1.0, 1.0, 2.0

# Inlier points (on the circle with small noise).
n_inliers = 40
theta_inliers = np.random.uniform(0, 2 * np.pi, n_inliers)
noise_inliers = np.random.normal(0, 0.1, n_inliers)
inlier_x = true_cx + (true_r + noise_inliers) * np.cos(theta_inliers)
inlier_y = true_cy + (true_r + noise_inliers) * np.sin(theta_inliers)

# Outlier points (scattered far from the circle).
n_outliers = 10
outlier_x = np.random.uniform(-4, 7, n_outliers)
outlier_y = np.random.uniform(-4, 7, n_outliers)

# Combine all points.
all_x = np.concatenate([inlier_x, outlier_x])
all_y = np.concatenate([inlier_y, outlier_y])
points = jnp.stack([all_x, all_y], axis=-1)
n_points = len(points)

# Track which points are outliers for visualization.
is_outlier = np.array([False] * n_inliers + [True] * n_outliers)

print(
    f"Generated {n_inliers} inliers and {n_outliers} outliers ({n_outliers / n_points * 100:.0f}% outliers)"
)
print(f"True circle: center=({true_cx}, {true_cy}), radius={true_r}")

## Standard least squares baseline

First, solve the unweighted problem so we can compare:


In [ ]:
class CircleVar(
    jaxls.Var[jax.Array], default_factory=lambda: jnp.array([0.0, 0.0, 1.0])
):
    """Circle parameters: [cx, cy, r]."""


@jaxls.Cost.factory
def circle_residual(
    vals: jaxls.VarValues,
    circle: CircleVar,
    point: jax.Array,
) -> jax.Array:
    """Plain 2D residual: error vector from closest circle point to observed point."""
    params = vals[circle]
    cx, cy, r = params[0], params[1], params[2]
    diff = point - jnp.array([cx, cy])
    dist = jnp.sqrt(jnp.sum(diff ** 2) + 1e-8)
    direction = diff / dist
    return (dist - r) * direction

In [ ]:
circle_var = CircleVar(id=0)

# Initial guess: centroid of points, average distance as radius.
centroid = jnp.mean(points, axis=0)
avg_dist = jnp.mean(jnp.sqrt(jnp.sum((points - centroid) ** 2, axis=-1)))
initial_params = jnp.array([centroid[0], centroid[1], avg_dist])

costs_standard = [
    circle_residual(CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points)
]
initial_vals = jaxls.VarValues.make([circle_var.with_value(initial_params)])
problem_standard = jaxls.LeastSquaresProblem(costs_standard, [circle_var]).analyze()
solution_standard = problem_standard.solve(initial_vals)

params_standard = solution_standard[circle_var]
print(
    f"Standard LS result: center=({params_standard[0]:.3f}, {params_standard[1]:.3f}), radius={params_standard[2]:.3f}"
)
print(f"True parameters:    center=({true_cx:.3f}, {true_cy:.3f}), radius={true_r:.3f}")

## Adaptive IRLS via `irls_weight_fn`

The adaptive factories work exactly like their fixed-parameter counterparts but
**estimate the noise scale from the data at every iteration**:

| Factory | Weight formula (normalised residual $u = r/\hat{\sigma}$) | Default $k$ |
|---------|-------------------------------------------------------------|-------------|
| `irls_huber_adaptive(k)` | $w = 1$ if $|u| \le k$, else $w = k/|u|$ | 1.345 |
| `irls_cauchy_adaptive(k)` | $w = 1/(1+(u/k)^2)$ | 2.385 |
| `irls_tukey_adaptive(k)` | $w = (1-(u/k)^2)^2$ if $|u|\le k$, else $0$ | 4.685 |

The default $k$ values are chosen so that each estimator achieves **95 % asymptotic efficiency**
relative to ordinary least squares under Gaussian noise—a standard rule of thumb.

Because the scale is estimated from the group's residuals at each step, you do **not** need
to choose a threshold in physical units.  The `irls_weight_fn` receives a
`(count, residual_flat_dim)` array and returns a matching weight array.

In [ ]:
def make_adaptive_robust_circle_cost(irls_weight_fn):
    """Return a Cost factory that applies the given adaptive IRLS weight function."""

    @jaxls.Cost.factory(irls_weight_fn=irls_weight_fn)
    def robust_circle_residual(
        vals: jaxls.VarValues,
        circle: CircleVar,
        point: jax.Array,
    ) -> jax.Array:
        """Plain geometric residual – adaptive weighting is handled by irls_weight_fn."""
        params = vals[circle]
        cx, cy, r = params[0], params[1], params[2]
        diff = point - jnp.array([cx, cy])
        dist = jnp.sqrt(jnp.sum(diff ** 2) + 1e-8)
        direction = diff / dist
        return (dist - r) * direction

    return robust_circle_residual


def compute_scalar_weights(params: jax.Array, points: jax.Array, weight_fn) -> jax.Array:
    """Compute a single scalar weight per point for visualisation.

    weight_fn follows the irls_weight_fn group signature: (count, dim) -> (count, dim).
    We reduce the per-element weights to per-point scalars by taking the mean.
    """
    cx, cy, r = params[0], params[1], params[2]
    dists = jnp.sqrt((points[:, 0] - cx) ** 2 + (points[:, 1] - cy) ** 2 + 1e-8)
    residuals = (dists - r)[:, None]  # shape (n, 1) — group with 1-D residuals
    weights = weight_fn(residuals)    # shape (n, 1)
    return weights[:, 0]              # shape (n,)

## Robust fitting with Cauchy adaptive weights

A single `solve()` call — no manual tuning of the scale parameter:

In [ ]:
cauchy_adaptive_cost = make_adaptive_robust_circle_cost(jaxls.utils.irls_cauchy_adaptive())

costs_cauchy = [cauchy_adaptive_cost(CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points)]
problem_cauchy = jaxls.LeastSquaresProblem(costs_cauchy, [circle_var]).analyze()
solution_cauchy = problem_cauchy.solve(initial_vals)

params_cauchy = solution_cauchy[circle_var]
weights_cauchy = compute_scalar_weights(params_cauchy, points, jaxls.utils.irls_cauchy_adaptive())

print(
    f"Adaptive Cauchy:  center=({params_cauchy[0]:.3f}, {params_cauchy[1]:.3f}), radius={params_cauchy[2]:.3f}"
)
print(f"True parameters: center=({true_cx:.3f}, {true_cy:.3f}), radius={true_r:.3f}")
print(
    f"\nCenter error: {jnp.sqrt((params_cauchy[0] - true_cx) ** 2 + (params_cauchy[1] - true_cy) ** 2):.4f}"
)
print(f"Radius error:  {jnp.abs(params_cauchy[2] - true_r):.4f}")

In [ ]:
import plotly.graph_objects as go
from IPython.display import HTML


def make_circle_trace(cx, cy, r, name, color, dash="solid"):
    import numpy as _np
    theta = _np.linspace(0, 2 * _np.pi, 100)
    return go.Scatter(
        x=cx + r * _np.cos(theta),
        y=cy + r * _np.sin(theta),
        mode="lines",
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )

## Weight visualisation

Points are coloured by their adaptive Cauchy weight (blue = high weight / inlier,
red = low weight / outlier).  The scale σ is inferred from the residuals, so even
if the absolute residuals change between problems, the relative weighting remains
well-calibrated.

In [ ]:
fig_w = go.Figure()

fig_w.add_trace(go.Scatter(
    x=all_x, y=all_y,
    mode="markers",
    marker=dict(
        size=12,
        color=weights_cauchy,
        colorscale=[[0, "#F44336"], [1, "#2196F3"]],
        colorbar=dict(title="Weight", thickness=15),
        cmin=0, cmax=1,
    ),
    text=[f"Weight: {w:.3f}" for w in weights_cauchy],
    hovertemplate="(%{x:.2f}, %{y:.2f})<br>%{text}<extra></extra>",
    name="Points",
))
fig_w.add_trace(make_circle_trace(
    float(params_cauchy[0]), float(params_cauchy[1]), float(params_cauchy[2]),
    "Adaptive Cauchy fit", "#9C27B0",
))
fig_w.add_trace(make_circle_trace(true_cx, true_cy, true_r, "True circle", "#4CAF50", "dash"))

fig_w.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_w.update_yaxes(title_text="y")
fig_w.update_layout(
    title="Points Coloured by Adaptive Cauchy IRLS Weight",
    height=450,
    margin=dict(t=60, b=40, l=60, r=40),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
HTML(fig_w.to_html(full_html=False, include_plotlyjs="cdn"))

## Comparing all three adaptive M-estimators

Run all three adaptive factories and compare to the standard-LS baseline:


In [ ]:
results = {"Standard LS": params_standard}

for name, weight_fn in [
    ("Huber (adaptive)",  jaxls.utils.irls_huber_adaptive()),
    ("Cauchy (adaptive)", jaxls.utils.irls_cauchy_adaptive()),
    ("Tukey (adaptive)",  jaxls.utils.irls_tukey_adaptive()),
]:
    cost = make_adaptive_robust_circle_cost(weight_fn)
    costs = [cost(CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points)]
    problem = jaxls.LeastSquaresProblem(costs, [circle_var]).analyze()
    solution = problem.solve(initial_vals, verbose=False)
    results[name] = solution[circle_var]

print(f"{'Method':<22} {'cx':>8} {'cy':>8} {'r':>8} {'Center err':>12} {'Radius err':>12}")
print("-" * 76)
print(f"{'True':<22} {true_cx:>8.3f} {true_cy:>8.3f} {true_r:>8.3f} {'-':>12} {'-':>12}")
for name, params in results.items():
    cerr = float(jnp.sqrt((params[0] - true_cx) ** 2 + (params[1] - true_cy) ** 2))
    rerr = float(jnp.abs(params[2] - true_r))
    print(f"{name:<22} {float(params[0]):>8.3f} {float(params[1]):>8.3f} {float(params[2]):>8.3f} {cerr:>12.4f} {rerr:>12.4f}")

In [ ]:
from plotly.subplots import make_subplots

colors = {
    "Standard LS":        "#FF9800",
    "Huber (adaptive)":   "#2196F3",
    "Cauchy (adaptive)":  "#9C27B0",
    "Tukey (adaptive)":   "#00BCD4",
}

fig_all = go.Figure()

fig_all.add_trace(go.Scatter(
    x=all_x[~is_outlier], y=all_y[~is_outlier],
    mode="markers", marker=dict(size=8, color="#607D8B"), name="Inliers",
))
fig_all.add_trace(go.Scatter(
    x=all_x[is_outlier], y=all_y[is_outlier],
    mode="markers", marker=dict(size=10, color="#F44336", symbol="x"), name="Outliers",
))
fig_all.add_trace(make_circle_trace(true_cx, true_cy, true_r, "True", "#4CAF50", "dash"))

for name, params in results.items():
    fig_all.add_trace(make_circle_trace(
        float(params[0]), float(params[1]), float(params[2]), name, colors[name]
    ))

fig_all.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_all.update_yaxes(title_text="y")
fig_all.update_layout(
    title="Comparison of Adaptive M-Estimators for Circle Fitting",
    height=500,
    margin=dict(t=60, b=40, l=60, r=40),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
HTML(fig_all.to_html(full_html=False, include_plotlyjs="cdn"))

## Summary

Adaptive IRLS in jaxls requires only two steps:

1. **Choose a factory** from `jaxls.utils`:
   - `irls_huber_adaptive()` – Huber loss, MAD scale, 95 % efficiency default
   - `irls_cauchy_adaptive()` – Cauchy/Lorentzian, smooth down-weighting
   - `irls_tukey_adaptive()` – Tukey bisquare, hard rejection of gross outliers

2. **Pass it to `Cost.factory`**:
   ```python
   @jaxls.Cost.factory(irls_weight_fn=jaxls.utils.irls_cauchy_adaptive())
   def robust_residual(vals, var, data):
       return plain_geometric_residual(vals, var, data)
   ```

The solver calls `irls_weight_fn` with a `(count, residual_flat_dim)` array of **all
current residuals** for the cost group.  The adaptive factories estimate σ via MAD,
normalise each residual, and return weights in the same shape.  You never need to pick
a threshold in physical units.

### Adaptive vs fixed-parameter

| | Fixed (`irls_cauchy(c=0.5)`) | Adaptive (`irls_cauchy_adaptive()`) |
|---|---|---|
| Scale parameter | You choose, in residual units | Estimated from data each iteration |
| Tuning effort | Needs domain knowledge | None—use default `k` |
| Behaviour on different noise levels | Fixed threshold | Automatically re-scales |

For more details see {class}`jaxls.Var`, {class}`jaxls.Cost`, and {class}`jaxls.LeastSquaresProblem`.